<a href="https://colab.research.google.com/github/vvrgit/NLP-LAB/blob/main/TextClassification1DCNNPyTorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [32]:
# Step 1: Import libraries
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split

In [33]:
# Step 2: Sample data
texts = [
    "free money now",
    "call now free offer",
    "hello how are you",
    "let us meet tomorrow",
    "win cash prize",
    "are you available tomorrow"
]

labels = [1, 1, 0, 0, 1, 0]   # 1 = spam, 0 = ham

In [34]:
# Step 3: Convert text to BoW
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(texts).toarray()
y = labels

In [35]:
# Step 4: Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [36]:
# Step 5: Create Dataset class
class TextDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [37]:
train_loader = DataLoader(TextDataset(X_train, y_train), batch_size=2, shuffle=True)
test_loader = DataLoader(TextDataset(X_test, y_test), batch_size=2)

In [38]:
# Step 6: Define CNN Model (with hidden layer)
class CNNModel(nn.Module):
    def __init__(self, input_size):
        super().__init__()

        kernels=4
        kernel_size=3
        self.conv = nn.Conv1d(1, kernels, kernel_size)

        pool_size=2
        self.pool = nn.MaxPool1d(pool_size)

        # Calculate flattened size
        flattened_size = kernels * ((input_size - pool_size)//2)

        # Hidden layer
        hidden_neurons=8
        self.fc1 = nn.Linear(flattened_size, hidden_neurons)   # 8 neurons (you can change)

        # Output layer
        output_neurons=1
        self.fc2 = nn.Linear(hidden_neurons, output_neurons)

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = x.unsqueeze(1)              # (batch, 1, features)
        x = torch.relu(self.conv(x))
        x = self.pool(x)

        x = x.view(x.size(0), -1)       # Flatten

        x = torch.relu(self.fc1(x))     # Hidden layer + activation
        x = self.fc2(x)                 # Output layer

        return self.sigmoid(x)

In [39]:
model = CNNModel(input_size=X.shape[1])

In [40]:
# Step 7: Loss and Optimizer
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [41]:
# Step 8: Training
for epoch in range(10):
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        output = model(X_batch).squeeze()
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

Epoch 1, Loss: 0.6635
Epoch 2, Loss: 0.6426
Epoch 3, Loss: 0.6047
Epoch 4, Loss: 0.5855
Epoch 5, Loss: 0.5310
Epoch 6, Loss: 0.5318
Epoch 7, Loss: 0.4609
Epoch 8, Loss: 0.4425
Epoch 9, Loss: 0.3948
Epoch 10, Loss: 0.3573


In [42]:
# Flatten y_batch from test_loader to get all actual labels
# The current approach iterates over test_loader and prints predictions per batch.
# To get overall metrics, we need to collect all actuals and predictions.
# Let's re-run the prediction loop to collect all true labels and predictions

y_pred = []
y_actual = []

model.eval()
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        output = model(X_batch).squeeze()
        preds = (output > 0.5).int()
        y_pred.extend(preds.tolist())
        y_actual.extend(y_batch.tolist())


# Convert actual_labels from float to int if needed by sklearn metrics
y_actual = [int(y) for y in y_actual]
print(y_pred)

[0, 0]


In [43]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

print("\nConfusion Matrix:")
print(confusion_matrix(y_actual, y_pred))

print("\nClassification Report:")
print(classification_report(y_actual, y_pred))

print("\nAccuracy Score:", accuracy_score(y_actual, y_pred))


Confusion Matrix:
[[1 0]
 [1 0]]

Classification Report:
              precision    recall  f1-score   support

           0       0.50      1.00      0.67         1
           1       0.00      0.00      0.00         1

    accuracy                           0.50         2
   macro avg       0.25      0.50      0.33         2
weighted avg       0.25      0.50      0.33         2


Accuracy Score: 0.5


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
